# Profile the movie data

This notebook provides a repeatable **data-quality profiling recipe** for the local TMDB movie cache. It reads cached JSONL records, uses the project's normalizer, and never makes API requests or changes the source data.

## Profiling recipe

1. Load and normalize the cache using the same table definitions as the project.
2. Record each table's row count, columns, data types, missing values, and declared key quality.
3. Search for exact, key, and semantic duplicate patterns that require further investigation.
4. Compare the memory requirements of the original and proposed data types.
5. Identify unusual values and patterns that may require validation against the source data or an agreed business rule.

Run top to bottom. A value flagged as unusual is a prompt for investigation, not an instruction to delete or correct it.


In [2]:
from IPython.display import display
import pandas as pd

from streamvault.cache import load_cached_records
from streamvault.config import CONFIG
from streamvault.normalize import normalize_movie_records

records = list(load_cached_records(CONFIG.cache_file).values())
tables = normalize_movie_records(records)
movies = tables.movies

frames = {
    name: getattr(tables, name)
    for name in (
        "movies",
        "genres",
        "movie_genres",
        "companies",
        "movie_companies",
        "countries",
        "movie_origin_countries",
        "movie_production_countries",
        "languages",
        "movie_spoken_languages",
        "collections",
        "people",
        "cast_credits",
        "crew_credits",
    )
}

print(f"Cache: {CONFIG.cache_file}")
print(f"Cached movie records: {len(records):,}")
if not records:
    print("No cached movies yet. Run `uv run streamvault` and rerun this notebook.")


Cache: /home/faazeleishola/training/streamvault/data/raw/movies.jsonl
Cached movie records: 112


## 1. Table inventory and schema

Start with the grain and shape of each table. A link/bridge table has one row per relationship, so it can have more rows than the movie table.


In [3]:
inventory = pd.DataFrame(
    [
        {"table": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in frames.items()
    ]
)
display(inventory)

schema_rows = []
for table_name, frame in frames.items():
    for column in frame.columns:
        values = frame[column]
        schema_rows.append(
            {
                "table": table_name,
                "column": column,
                "dtype": str(values.dtype),
                "missing": int(values.isna().sum()),
                "missing_percent": (
                    round(values.isna().mean() * 100, 2) if len(frame) else pd.NA
                ),
                "distinct_non_missing": int(values.nunique(dropna=True)),
            }
        )

schema_profile = pd.DataFrame(schema_rows)

import ipywidgets as widgets

page_size = 15
page_count = (len(schema_profile) + page_size - 1) // page_size
if page_count == 0:
    print("No schema rows to display.")
else:
    page_state = {"index": 0}
    previous_button = widgets.Button(description="Previous", icon="chevron-left")
    next_button = widgets.Button(description="Next", icon="chevron-right")
    page_label = widgets.Label()
    page_output = widgets.Output()

    def render_schema_page():
        page_index = page_state["index"]
        start = page_index * page_size
        end = min(start + page_size, len(schema_profile))
        with page_output:
            page_output.clear_output(wait=True)
            display(schema_profile.iloc[start:end])
        page_label.value = (
            f"Page {page_index + 1} of {page_count} "
            f"(rows {start + 1}-{end} of {len(schema_profile)})."
        )
        previous_button.disabled = page_index == 0
        next_button.disabled = page_index == page_count - 1

    def show_previous_page(_):
        page_state["index"] -= 1
        render_schema_page()

    def show_next_page(_):
        page_state["index"] += 1
        render_schema_page()

    previous_button.on_click(show_previous_page)
    next_button.on_click(show_next_page)
    display(widgets.HBox([previous_button, page_label, next_button]))
    display(page_output)
    render_schema_page()


,table,rows,columns
0,movies,112,22
1,genres,16,2
2,movie_genres,320,2
3,companies,136,4
4,movie_companies,307,2
5,countries,12,2
6,movie_origin_countries,117,2
7,movie_production_countries,154,2
8,languages,27,3
9,movie_spoken_languages,219,2


Output()

### Initial observations from the table profile

- `adult`: All 112 movies are marked `False`, so this field is constant in the current cache.
- `budget`: All movies have a budget value, and 60 distinct budget values occur.
- `homepage`: Two homepage URLs occur more than once. This is repeated metadata, not necessarily evidence of a duplicate movie.
- `original_language`: English accounts for 110 movies; Japanese and Italian each account for one movie.
- `runtime`: Runtime has fewer distinct values than the number of movies, so repeated runtimes require inspection.
- `softcore`: All 112 movies are marked `False`, as is `adult`.
- `status`: All 112 movies are `Released`, so this column is constant in the current dataset.
- `video`: All 112 movies are marked `False`, so this field is also constant.
- `vote_average`: 106 of 112 values are distinct. The repeated values need to be inspected; equal floating-point values may come from genuine repeats or source rounding.
- In the language reference table, `language_name` and `language_english_name` often contain the same value, but they should not be removed without confirming the data is intended to support native-language display.

### Follow-up conclusions

- Repeated runtime and vote-average values have now been investigated below, but repetition alone does not establish that a movie is a duplicate.
- Constant fields such as `adult`, `softcore`, `status`, and `video` provide little discriminating value in this cache. The question of whether to remove or retain them should consider the needs of future datasets rather than only the current 112 rows.
- `original_language` is overwhelmingly English. It may be suitable for category encoding, but it should not be considered a low-variance feature without a predictive use case.
- The language-name columns is not different from the language name in english

### Adult flag

The `adult` column is constant in the current cache. Confirming this value helps distinguish a data-quality limitation from a simply low-variance field.

In [3]:
if movies.empty or "adult" not in movies.columns:
    print("No adult data available for profiling.")
else:
    display(movies["adult"].value_counts())

adult
False    112
Name: count, dtype: int64

### Budget distribution

This summary establishes the observed range and spread of movie budgets. Because every cached movie has a budget value, missingness is not an issue in this snapshot.

In [4]:
if movies.empty or "budget" not in movies.columns:
    print("No budget data available for profiling.")
else:
    display(movies["budget"].describe())

count    1.120000e+02
mean     1.084460e+08
std      8.278818e+07
min      1.200000e+06
25%      3.808750e+07
50%      9.100000e+07
75%      1.500000e+08
max      3.790000e+08
Name: budget, dtype: float64

### Original language

The count is dominated by English. This helps identify whether the cache is representative of a broader movie collection.

In [5]:
if movies.empty or "original_language" not in movies.columns:
    print("No original-language data available for profiling.")
else:
    display(movies["original_language"].value_counts())

original_language
en    110
ja      1
it      1
Name: count, dtype: int64

### Release status

A single value means that `status` cannot distinguish between movies in the current cache. The column may still be useful if future data includes other statuses.

In [6]:
if movies.empty or "status" not in movies.columns:
    print("No release-status data available for profiling.")
else:
    display(movies["status"].value_counts())

status
Released    112
Name: count, dtype: int64

### Collection membership

This count shows how many cached movies point to each collection. Collection IDs that are missing are not shown by this result and should be evaluated separately.

In [7]:
if movies.empty or "collection_id" not in movies.columns:
    print("No collection data available for profiling.")
else:
    display(movies["collection_id"].value_counts())

collection_id
1241      8
10        6
295       5
87359     5
8091      4
1570      4
10194     4
86311     4
2150      4
84        4
528       4
263       3
264       3
230       3
328       3
131292    3
119       3
121938    3
2344      3
304       3
8354      3
31562     3
2883      2
94032     1
468222    1
9743      1
134897    1
422837    1
2366      1
137696    1
8945      1
2980      1
Name: count, dtype: Int64

### Language reference-table coverage

The output below confirms the native-language and English-language names for each language code. The `missing` frame is available for checking whether a native name is absent.

In [8]:
languages = frames.get("languages")
missing = languages[languages["language_name"].isna()]
languages

,language_code,language_name,language_english_name
0,en,English,English
1,zh,普通话,Mandarin
2,it,Italiano,Italian
3,de,Deutsch,German
4,la,Latin,Latin
5,es,Español,Spanish
6,fr,Français,French
7,tr,Türkçe,Turkish
8,el,ελληνικά,Greek
9,ru,Pусский,Russian


### Missing person departments

The first output identifies person-reference rows without a `known_for_department`. Inspect the corresponding source credits before deciding whether a missing value is intentional.

In [9]:
people = frames.get("people")
missing = people[people["known_for_department"].isna()]
missing

,person_id,adult,gender,known_for_department,name,original_name,profile_path
17473,8168,False,2,NaN,Brian Magerkurth,Brian Magerkurth,NaN


### Known-for department distribution

This summary shows how people in the reference table are classified across departments.

In [10]:
people["known_for_department"].value_counts()

known_for_department
Acting               6198
Crew                 3215
Visual Effects       2457
Production           1935
Art                  1766
Sound                1536
Costume & Make-Up    1287
Camera               1118
Directing             827
Lighting              687
Editing               527
Writing               287
Creator                 3
Name: count, dtype: int64

### Investigation of repeated vote averages

This output lists movies that share a vote-average value. Repeated values can result from source rounding or genuine ties; they do not by themselves indicate duplicate movies.

In [11]:
movies = frames.get("movies")

if movies.empty or "vote_average" not in movies.columns:
    print("No vote-average data available for profiling.")
else:
    vote_avg_counts = movies["vote_average"].value_counts()
    dup_avgs = vote_avg_counts[vote_avg_counts > 1]

    for vote_avg in dup_avgs.index:
        frame_x = movies.loc[
            movies["vote_average"] == vote_avg,
            ["title", "vote_average", "vote_count"]
        ]
        print(f"\nvote_average = {vote_avg} ({dup_avgs[vote_avg]} movies)")
        display(frame_x)



vote_average = 8.451 (2 movies)


,title,vote_average,vote_count
14,GoodFellas,8.451,15041
18,The Lord of the Rings: The Fellowship of the Ring,8.451,28890



vote_average = 7.9 (2 movies)


,title,vote_average,vote_count
20,Return of the Jedi,7.9,17566
91,Indiana Jones and the Last Crusade,7.9,11543



vote_average = 8.247 (2 movies)


,title,vote_average,vote_count
27,Avengers: Endgame,8.247,29384
51,Avengers: Infinity War,8.247,33418


... (2 outputs left out)


vote_average = 7.818 (2 movies)


,title,vote_average,vote_count
47,Die Hard,7.818,12876
86,Toy Story 3,7.818,16456



vote_average = 8.2 (2 movies)


,title,vote_average,vote_count
64,Alien,8.2,17148
75,Memento,8.2,17054


### Investigation of repeated runtimes

Movies with the same runtime are grouped for review. Similar lengths can be coincidental, and these groups are not evidence of duplicate or incorrect records.

In [12]:
if movies.empty or "runtime" not in movies.columns:
    print("No runtime data available for profiling.")
else:
    runtime_counts = movies["runtime"].value_counts()
    dup_runtimes = runtime_counts[runtime_counts > 1]

    for runtime in dup_runtimes.index:
        frame_x = movies.loc[
            movies["runtime"] == runtime,
            ["title", "runtime", "vote_average", "vote_count", "tagline", "release_date"]
        ]
        print(f"\nruntime = {runtime} minutes ({dup_runtimes[runtime]} movies)")
        display(frame_x)



runtime = 92 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
5,Jurassic Park III,92,6.179,8195,"This time, it's not just a walk in the park!",2001-07-18
26,Toy Story 2,92,7.623,15594,The toys are back!,1999-10-30
76,Shrek 2,92,7.333,13982,What happens after happily ever after?,2004-05-19
88,"Monsters, Inc.",92,7.860,20528,It's nothing personal. It's just their job.,2001-11-01



runtime = 124 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
8,Iron Man 2,124,6.856,23248,"It's not the armor that makes the hero, but th...",2010-04-28
13,Die Hard 2,124,6.971,6718,Die harder.,1990-07-03
49,The Empire Strikes Back,124,8.392,19137,The Star Wars saga continues.,1980-05-20
85,Jaws,124,7.687,12039,The terrifying motion picture from the terrify...,1975-06-20



runtime = 136 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
31,Star Wars: Episode I - The Phantom Menace,136,6.582,16435,Every saga has a beginning.,1999-05-19
35,The Matrix,136,8.263,29272,Believe the unbelievable.,1999-03-31
73,Pirates of the Caribbean: On Stranger Tides,136,6.576,15449,Live forever or die trying.,2011-05-15
99,Kill Bill: Vol. 2,136,7.872,15687,The bride is back for the final cut.,2004-04-16


... (58 outputs left out)


runtime = 140 minutes (2 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
104,Star Wars: Episode III - Revenge of the Sith,140,7.479,15516,Every story has a hero. Every hero has a desti...,2005-05-17
105,Batman Begins,140,7.733,23544,Evil fears the knight.,2005-06-10



runtime = 109 minutes (2 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
110,Terminator 3: Rise of the Machines,109,6.216,7641,The machines will rise.,2003-07-02
111,Alien Resurrection,109,6.170,5877,It's already too late.,1997-11-12


In [13]:
key_columns = {
    "movies": ["movie_id"],
    "genres": ["genre_id"],
    "movie_genres": ["movie_id", "genre_id"],
    "companies": ["company_id"],
    "movie_companies": ["movie_id", "company_id"],
    "countries": ["country_code"],
    "movie_origin_countries": ["movie_id", "country_code"],
    "movie_production_countries": ["movie_id", "country_code"],
    "languages": ["language_code"],
    "movie_spoken_languages": ["movie_id", "language_code"],
    "collections": ["collection_id"],
    "people": ["person_id"],
    "cast_credits": ["credit_id"],
    "crew_credits": ["credit_id"],
}

key_quality_rows = []
for table_name, columns in key_columns.items():
    frame = frames[table_name]
    key_quality_rows.append(
        {
            "table": table_name,
            "key": " + ".join(columns),
            "rows_with_null_key": int(frame[columns].isna().any(axis=1).sum()),
            "duplicate_key_rows": int(frame.duplicated(subset=columns).sum()),
        }
    )

key_quality = pd.DataFrame(key_quality_rows)
display(key_quality)


,table,key,rows_with_null_key,duplicate_key_rows
0,movies,movie_id,0,0
1,genres,genre_id,0,0
2,movie_genres,movie_id + genre_id,0,0
3,companies,company_id,0,0
4,movie_companies,movie_id + company_id,0,0
5,countries,country_code,0,0
6,movie_origin_countries,movie_id + country_code,0,0
7,movie_production_countries,movie_id + country_code,0,0
8,languages,language_code,0,0
9,movie_spoken_languages,movie_id + language_code,0,0


## 2. Duplicate detection

This section checks exact duplicate rows and semantic duplicate candidates. Exact duplicates indicate a source-cache or pipeline anomaly; semantic duplicates require manual review because shared titles and release years do not prove that two records describe the same film.


### Exact duplicates

The cache loader indexes successful responses by `movie_id`, and the normalizer rejects duplicate IDs. This check therefore separates raw-cache line/object repetition from exact repeated rows in the normalized movie table. Identical raw objects for different IDs are not possible by this grain, but an exact-line/object audit remains useful as a pipeline regression check.


In [14]:
import json
from collections import Counter

raw_lines = [
    line.strip()
    for line in CONFIG.cache_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
line_counts = Counter(raw_lines)
identical_line_groups = {line: count for line, count in line_counts.items() if count > 1}

canonical_objects = Counter()
for line in raw_lines:
    record = json.loads(line)
    canonical_objects[json.dumps(record, sort_keys=True, ensure_ascii=False)] += 1
duplicate_object_groups = sum(count - 1 for count in canonical_objects.values() if count > 1)
normalized_exact_mask = movies.duplicated(keep=False)
normalized_exact_suspects = movies.loc[normalized_exact_mask].copy()

print(f"Identical raw JSON lines: {sum(count - 1 for count in identical_line_groups.values())}")
print(f"Repeated raw JSON objects: {duplicate_object_groups}")
print(f"Exact duplicate normalized movie rows: {int(normalized_exact_mask.sum())}")
display(normalized_exact_suspects)


Identical raw JSON lines: 0
Repeated raw JSON objects: 0
Exact duplicate normalized movie rows: 0


,movie_id,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,...,revenue,runtime,softcore,status,tagline,title,video,vote_average,vote_count,collection_id


### Semantic duplicates

Two different IDs can describe the same real film, especially after title or release reissue variations. The check below uses a normalized title and release year as candidate keys only; matches require manual review and are never automatically removed.


In [15]:
semantic_review = movies.copy()
if "title" in semantic_review.columns:
    normalized_title = (
        semantic_review["title"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )
else:
    semantic_review["title"] = pd.Series(pd.NA, index=semantic_review.index, dtype="string")
    normalized_title = pd.Series(pd.NA, index=semantic_review.index, dtype="string")

if "release_date" in semantic_review.columns:
    release_year = semantic_review["release_date"].dt.year
else:
    release_year = pd.Series(pd.NA, index=semantic_review.index, dtype="Int64")

semantic_review["normalized_title"] = normalized_title
semantic_review["release_year"] = release_year
has_semantic_key = (
    semantic_review["normalized_title"].notna()
    & semantic_review["normalized_title"].ne("")
    & semantic_review["release_year"].notna()
).fillna(False)
eligible_rows = semantic_review.loc[has_semantic_key]
candidate_mask = eligible_rows.duplicated(
    subset=["normalized_title", "release_year"], keep=False
)
semantic_suspects = eligible_rows.loc[candidate_mask].copy()

semantic_candidate_groups = (
    semantic_suspects.groupby(
        ["normalized_title", "release_year"], as_index=False
    )
    .agg(
        movie_count=("movie_id", "size"),
        movie_ids=("movie_id", list),
        titles=("title", list),
    )
    .sort_values(["movie_count", "normalized_title"], ascending=[False, True])
)

review_columns = [
    column
    for column in (
        "movie_id",
        "title",
        "original_title",
        "release_date",
        "imdb_id",
        "collection_id",
    )
    if column in semantic_suspects.columns
]
print(
    f"{len(semantic_suspects)} rows in {len(semantic_candidate_groups)} "
    f"candidate groups; {int(has_semantic_key.sum())} rows eligible; "
    f"{int((~has_semantic_key).sum())} rows missing title or release year."
)
display(semantic_candidate_groups)
display(
    semantic_suspects[
        review_columns + ["normalized_title", "release_year"]
    ].sort_values(["normalized_title", "release_year", "movie_id"])
)


0 rows in 0 candidate groups; 112 rows eligible; 0 rows missing title or release year.


,normalized_title,release_year,movie_count,movie_ids,titles


,movie_id,title,original_title,release_date,imdb_id,collection_id,normalized_title,release_year


## 3. Text format and placeholders

These checks identify formatting anomalies and source placeholders for manual review. Repeated values can be legitimate: for example, many movies share a homepage or collection. A warning here does not remove or change a record.


In [16]:
placeholder_pattern = r"^(unknown|n/?a|none|null|not available|-+)$"
text_format_rows = []
for table_name, frame in frames.items():
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        values = frame[column]
        as_string = values.astype("string")
        blank_count = int(as_string.str.strip().eq("").fillna(False).sum())
        whitespace_count = int((as_string.notna() & as_string.ne(as_string.str.strip())).sum())
        placeholder_count = int(as_string.str.strip().str.casefold().str.fullmatch(placeholder_pattern).fillna(False).sum())
        text_format_rows.append({
            "table": table_name,
            "column": column,
            "blank_or_whitespace_only": blank_count,
            "leading_or_trailing_whitespace": whitespace_count,
            "placeholder_like": placeholder_count,
        })

text_format_profile = pd.DataFrame(text_format_rows)
display(text_format_profile)

if "homepage" in movies.columns:
    homepage_counts = movies["homepage"].dropna().astype("string").value_counts()
    repeated_homepages = homepage_counts[homepage_counts > 1].rename("movie_count").reset_index().rename(columns={"index": "homepage"})
    print("Repeated homepage URLs:")
    display(repeated_homepages)

language_name_columns = [column for column in ("language_name", "language_english_name") if column in tables.languages.columns]
if language_name_columns:
    unknown_languages = tables.languages.loc[
        tables.languages[language_name_columns].isna().any(axis=1)
    ]
    print("Languages missing a native or English name:")
    display(unknown_languages)

url_like_columns = [column for column in ("homepage", "poster_path", "backdrop_path", "profile_path") if column in movies.columns]
malformed_url_rows = []
for column in url_like_columns:
    values = movies[column].dropna().astype("string")
    malformed = values.loc[~values.str.match(r"^(https?://|/)")]
    if not malformed.empty:
        malformed_url_rows.append({"column": column, "unexpected_format_count": len(malformed), "examples": malformed.head(5).tolist()})
print("Unexpected URL/path formats:")
display(pd.DataFrame(malformed_url_rows))


,table,column,blank_or_whitespace_only,leading_or_trailing_whitespace,placeholder_like
0,movies,backdrop_path,0,0,0
1,movies,homepage,0,0,0
2,movies,imdb_id,0,0,0
3,movies,original_language,0,0,0
4,movies,original_title,0,0,0
5,movies,overview,0,0,0
6,movies,poster_path,0,0,0
7,movies,status,0,0,0
8,movies,tagline,0,0,0
9,movies,title,0,0,0


Repeated homepage URLs:


,homepage,movie_count
0,http://www.lordoftherings.net/,2
1,http://www.thehobbit.com/,2


Languages missing a native or English name:


,language_code,language_name,language_english_name
14,xh,NaN,Xhosa
24,ne,NaN,Nepali


Unexpected URL/path formats:


""


## 4. Valid range profiling

This section reports invariant failures and descriptive range observations. Mathematical rules such as negative vote counts or a vote average outside 0–10 should stop the pipeline if they indicate corrupted data. Observed bounds are current evidence, not permanent business rules; proposals should be recorded in `DECISIONS.md` before becoming fail-fast checks.


In [17]:
range_rules = {
    "runtime": {"minimum": 0, "maximum": None, "proposed_minimum": 1, "proposed_maximum": 600},
    "budget": {"minimum": 0, "maximum": None, "proposed_minimum": 0, "proposed_maximum": None},
    "revenue": {"minimum": 0, "maximum": None, "proposed_minimum": 0, "proposed_maximum": None},
    "popularity": {"minimum": 0, "maximum": None, "proposed_minimum": 0, "proposed_maximum": None},
    "vote_count": {"minimum": 0, "maximum": None, "proposed_minimum": 0, "proposed_maximum": None},
    "vote_average": {"minimum": 0, "maximum": 10, "proposed_minimum": 0, "proposed_maximum": 10},
}

range_rows = []
for column, rule in range_rules.items():
    if column not in movies.columns:
        continue
    values = pd.to_numeric(movies[column], errors="coerce")
    observed = values.dropna()
    below = values.lt(rule["minimum"]) if rule["minimum"] is not None else pd.Series(False, index=values.index)
    above = values.gt(rule["maximum"]) if rule["maximum"] is not None else pd.Series(False, index=values.index)
    range_rows.append({
        "field": column,
        "dtype": str(movies[column].dtype),
        "minimum_observed": observed.min() if not observed.empty else pd.NA,
        "maximum_observed": observed.max() if not observed.empty else pd.NA,
        "missing_values": int(values.isna().sum()),
        "zero_values": int(values.eq(0).sum()),
        "negative_or_below_minimum": int(below.sum()),
        "above_maximum": int(above.sum()),
    })

range_profile = pd.DataFrame(range_rows)
display(range_profile)

release_dates = movies["release_date"] if "release_date" in movies.columns else pd.Series(dtype="datetime64[us]")
release_profile = pd.DataFrame([{
    "field": "release_date",
    "dtype": str(release_dates.dtype),
    "minimum_observed": release_dates.min() if release_dates.notna().any() else pd.NA,
    "maximum_observed": release_dates.max() if release_dates.notna().any() else pd.NaT,
    "missing_values": int(release_dates.isna().sum()),
    "invalid_dates_reaching_profile": 0,
}])
display(release_profile)

print("Range violations are reported as warnings. Invalid normalized dates should remain a fail-fast normalizer check.")


,field,dtype,minimum_observed,maximum_observed,missing_values,zero_values,negative_or_below_minimum,above_maximum
0,runtime,int64,8.100000e+01,2.020000e+02,0,0,0,0
1,budget,int64,1.200000e+06,3.790000e+08,0,0,0,0
2,revenue,int64,7.390108e+06,2.942650e+09,0,0,0,0
3,popularity,float64,4.583000e-01,8.911770e+01,0,0,0,0
4,vote_count,int64,4.136000e+03,4.192500e+04,0,0,0,0
5,vote_average,float64,5.752000e+00,8.734000e+00,0,0,0,0


,field,dtype,minimum_observed,maximum_observed,missing_values,invalid_dates_reaching_profile
0,release_date,datetime64[us],1966-12-23,2019-06-19,0,0


Range violations are reported as warnings. Invalid normalized dates should remain a fail-fast normalizer check.


## 5. Data Types: Validation, Downcasting and Encoding

This section benchmarks candidate dtype changes incrementally and preserves the normalized `movies` DataFrame. Each conversion is applied to a copy only after its observed range and missing values are checked.


In [18]:
from pandas.api.types import is_integer_dtype

if movies.empty:
    print("No movie rows available to benchmark.")
else:
    original_movies = movies.copy(deep=True)
    benchmark_movies = movies.copy(deep=True)
    memory_stages = []

    def record_memory(stage, frame):
        memory_bytes = int(frame.memory_usage(index=True, deep=True).sum())
        previous_bytes = memory_stages[-1]["memory_bytes"] if memory_stages else None
        baseline_bytes = memory_stages[0]["memory_bytes"] if memory_stages else memory_bytes
        memory_stages.append(
            {
                "stage": stage,
                "memory_bytes": memory_bytes,
                "memory_kib": round(memory_bytes / 1024, 2),
                "saved_bytes_vs_previous": (
                    pd.NA if previous_bytes is None else previous_bytes - memory_bytes
                ),
                "saved_percent_vs_baseline": round(
                    (baseline_bytes - memory_bytes) / baseline_bytes * 100, 2
                ) if baseline_bytes else 0.0,
            }
        )

    record_memory("Baseline: normalized movies", benchmark_movies)

    dtype_change_rows = []
    integer_columns = [
        column
        for column in benchmark_movies.columns
        if is_integer_dtype(benchmark_movies[column].dtype)
    ]
    for column in integer_columns:
        before = benchmark_movies[column]
        observed = before.dropna()
        if observed.empty:
            continue

        downcast_kind = "unsigned" if bool(observed.ge(0).all()) else "integer"
        after = pd.to_numeric(before, downcast=downcast_kind)
        if not before.astype("string").equals(after.astype("string")):
            raise AssertionError(f"Downcasting changed values in {column!r}.")

        benchmark_movies[column] = after
        dtype_change_rows.append(
            {
                "column": column,
                "before_dtype": str(before.dtype),
                "after_dtype": str(after.dtype),
                "minimum": observed.min(),
                "maximum": observed.max(),
                "missing_values": int(before.isna().sum()),
                "downcast_kind": downcast_kind,
            }
        )
        record_memory(f"After integer downcast: {column}", benchmark_movies)

    category_columns = [
        column
        for column in ("original_language", "status")
        if column in benchmark_movies.columns
    ]
    for column in category_columns:
        before = benchmark_movies[column]
        after = before.astype("category")
        if not before.astype("string").equals(after.astype("string")):
            raise AssertionError(f"Categorical encoding changed values in {column!r}.")

        benchmark_movies[column] = after
        dtype_change_rows.append(
            {
                "column": column,
                "before_dtype": str(before.dtype),
                "after_dtype": str(after.dtype),
                "minimum": pd.NA,
                "maximum": pd.NA,
                "missing_values": int(before.isna().sum()),
                "downcast_kind": "category",
            }
        )
        record_memory(f"After categorical encoding: {column}", benchmark_movies)

    if not movies.equals(original_movies):
        raise AssertionError("The experiment unexpectedly changed the normalized movies DataFrame.")

    print("Memory includes the DataFrame index and deep memory for values.")
    print("Positive saved-byte values indicate a reduction from the preceding stage.")
    display(pd.DataFrame(memory_stages))
    display(pd.DataFrame(dtype_change_rows))

    raw_release_dates_by_id = {
        int(record["id"]): record.get("release_date")
        for record in records
    }
    source_release_dates = movies["movie_id"].map(raw_release_dates_by_id).astype("string")
    parsed_release_dates = pd.to_datetime(
        source_release_dates,
        format="%Y-%m-%d",
        errors="coerce",
    )
    invalid_date_mask = source_release_dates.notna() & parsed_release_dates.isna()
    if invalid_date_mask.any():
        bad_movie_ids = movies.loc[invalid_date_mask, "movie_id"].tolist()
        raise ValueError(f"Invalid source release dates for movie IDs: {bad_movie_ids}")

    dates_match_normalized = (
        parsed_release_dates.eq(movies["release_date"])
        | (parsed_release_dates.isna() & movies["release_date"].isna())
    )
    if not bool(dates_match_normalized.fillna(False).all()):
        raise AssertionError("Notebook-parsed release dates differ from normalized values.")

    date_representation = pd.DataFrame(
        [
            {
                "representation": "Source date strings",
                "dtype": str(source_release_dates.dtype),
                "memory_bytes": int(source_release_dates.memory_usage(index=False, deep=True)),
            },
            {
                "representation": "Parsed datetime",
                "dtype": str(parsed_release_dates.dtype),
                "memory_bytes": int(parsed_release_dates.memory_usage(index=False, deep=True)),
            },
        ]
    )
    date_representation["memory_kib"] = (
        date_representation["memory_bytes"] / 1024
    ).round(2)
    print("Release-date representation comparison (column only):")
    display(date_representation)
    print(
        f"Parsed {int(source_release_dates.notna().sum()):,} non-missing source dates; "
        "all parsed values match the normalized dates."
    )
    print(
        "Runtime is not benchmarked: use a larger, representative cache and repeated "
        "timings before drawing runtime conclusions."
    )


Memory includes the DataFrame index and deep memory for values.
Positive saved-byte values indicate a reduction from the preceding stage.


,stage,memory_bytes,memory_kib,saved_bytes_vs_previous,saved_percent_vs_baseline
0,Baseline: normalized movies,140626,137.33,<NA>,0.00
1,After integer downcast: movie_id,140178,136.89,448,0.32
2,After integer downcast: budget,139730,136.46,448,0.64
3,After integer downcast: revenue,139282,136.02,448,0.96
4,After integer downcast: runtime,138498,135.25,784,1.51
5,After integer downcast: vote_count,137826,134.60,672,1.99
6,After integer downcast: collection_id,137378,134.16,448,2.31
7,After categorical encoding: original_language,131931,128.84,5447,6.18
8,After categorical encoding: status,125716,122.77,6215,10.60


,column,before_dtype,after_dtype,minimum,maximum,missing_values,downcast_kind
0,movie_id,int64,uint32,11,301528,0,unsigned
1,budget,int64,uint32,1200000,379000000,0,unsigned
2,revenue,int64,uint32,7390108,2942649778,0,unsigned
3,runtime,int64,uint8,81,202,0,unsigned
4,vote_count,int64,uint16,4136,41925,0,unsigned
5,collection_id,Int64,UInt32,10,468222,16,unsigned
6,original_language,str,category,<NA>,<NA>,0,category
7,status,str,category,<NA>,<NA>,0,category


Release-date representation comparison (column only):


,representation,dtype,memory_bytes,memory_kib
0,Source date strings,string,6608,6.45
1,Parsed datetime,datetime64[us],896,0.88


Parsed 112 non-missing source dates; all parsed values match the normalized dates.
Runtime is not benchmarked: use a larger, representative cache and repeated timings before drawing runtime conclusions.


## 6. Conclusions and limitations

### What this profile supports

- The current normalized snapshot contains **112 movies**. The inventory describes the grain and shape of the movie, genre, company, collection, language, person, cast, and crew tables. All profiled key fields are populated and all declared primary/composite keys tested in the notebook are unique in this snapshot.
- A number of columns were found to have no variance.
- The duplicate checks found **no identical raw JSON lines or objects, no exact duplicate normalized movie rows, and no title/release-year semantic duplicate candidates**. The semantic check is a candidate screen, not proof that no real-world duplicate exists.
- The tested numeric movie fields have no missing values, zeros, negative values, or observed range violations in this cache. All 112 release dates parsed and matched the normalized values, covering 1966-12-23 through 2019-06-19.
- Text checks found no unexpected URL/path formats. Two homepage URLs are shared by pairs of movies; that is repeated metadata, not evidence that the films are duplicates. Two language rows lack a native-language name (`xh` and `ne`), which merits source/reference review.
- The copy-only dtype experiment preserved the normalized `movies` frame and the tested values. For this 112-row frame, the candidate downcasts/category encodings reduced measured memory from 140,626 to 125,716 bytes (about 10.6%). This is a small-snapshot memory benchmark, not evidence of faster runtime or production-scale benefit.

### What this profile cannot determine

- A clean result for these checks does not prove that every source value is correct. Plausible but inaccurate budgets, revenues, dates, credits, or identifiers can pass range and duplicate checks.
- The sample is small and narrowly distributed: all movies are marked `Released`; `adult`, `softcore`, and `video` are constant; 110 of 112 original-language values are English. These fields cannot demonstrate broad population coverage or category behavior here.
- Repeated vote averages or runtimes are not duplicate evidence by themselves. The title/year semantic screen can miss alternate titles, reissues, or wrong source identifiers and can require manual review when it finds candidates.
- Missing person-department and language-name metadata, source freshness, full cast/crew correctness, and completeness of collection membership require source-level or domain review beyond this profile.
- Observed ranges are descriptive for this cache, not universal business rules. The experiment did not benchmark runtime and should not be used to set permanent validation thresholds without broader evidence.

**Relevance:** this profile provides evidence that the current normalized snapshot is structurally consistent across the checks performed, identifies a few metadata and representativeness limitations, and documents a cautious memory experiment. It does not certify the source as error-free or establish whether the fields are useful for prediction or licensing decisions.